# 06 — Simulation

The baseline gives one seat number. This asks how much of that number is the
swing assumption rather than anything about West Bengal.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
from src.features import load_seat_features
from src.models import statewide_swing, target, uniform_swing_prediction
from src.simulate import LEAD_BETA, run, sensitivity, summarise

seats = load_seat_features()
actual = int(target(seats).sum())
print(f"BJP actually won {actual} of {len(seats)}")

## The swing wasn't independent of the seat

Phase 3 treated each seat's deviation from the statewide swing as noise. It
isn't. Regress each seat's own swing on the 2021 TMC lead and the slope is
positive: BJP gained most where TMC had most to defend.

In [ ]:
local = (seats.swing_bjp - seats.swing_tmc) / 2
beta, alpha = np.polyfit(seats.tmc_lead_2021, local, 1)
resid = local - (alpha + beta * seats.tmc_lead_2021)
print(f"local_swing = {alpha:.2f} + {beta:.4f} * tmc_lead_2021")
print(f"  R^2 {1 - resid.var()/local.var():.3f}, residual sd {resid.std(ddof=2):.2f}")

tip = 2 * statewide_swing(seats)
for lo, hi, label in [(-100, 0, "BJP already held"),
                      (0, tip, "TMC lead under the tipping line"),
                      (tip, 100, "TMC lead over it")]:
    m = local[(seats.tmc_lead_2021 > lo) & (seats.tmc_lead_2021 <= hi)]
    print(f"  {label:34s} n={len(m):3d}  mean swing {m.mean():5.2f}")

5.9 points where BJP already held the seat, 9.4 where TMC was safe. R² is only
0.11, so it's a weak relationship in variance terms — but it acts exactly on
the seats the baseline never flips, so it moves the seat total a long way.

## What each assumption is worth

In [ ]:
sensitivity(seats)

Plain uniform swing calls 198. Adding the lead-scaled term takes it to 204,
against an actual 207 — most of the baseline's 9-seat shortfall was this one
effect.

Then the spread. Local noise alone puts 90% of draws between about 162 and
211. Letting the statewide swing itself be uncertain, at a standard deviation
of 2 points, roughly doubles that range.

That 2 points is an assumption, not a measurement. This data has one election
and cannot tell you how wrong a forecast of the swing would have been.

## The distribution

In [ ]:
counts = run(seats, save=True)
summarise(counts, actual=actual)

![seat distribution](../figures/05_seat_distribution.png)

The actual result sits around the 75th percentile. The simulation is
systematically a little pessimistic about BJP, and the reason is in the test
suite: mean-zero seat noise *lowers* the expected total, because more seats
sit just inside the tipping line than just outside, so noise knocks more out
than it brings in.

## What this is not

A forecast. Every parameter here was fitted on the election it then predicts,
including the lead-scaling slope. The interval says how sensitive the seat
total is to the swing, given 2021 — not how uncertain anyone should have been
in advance.